In [4]:
import pandas as pd
import os
import multiprocessing
import functions_stanza

/home/christopher/projects/corpus_tryout/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-09 16:01:00 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-03-09 16:01:00 INFO: Downloaded file to /home/christopher/stanza_resources/resources.json
2026-03-09 16:01:01 INFO: Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined                  |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | com

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
dfs = []

for file in os.listdir("gerunzius_download"):
    csv = pd.read_csv(f"gerunzius_download/{file}", sep=";", header=None)
    csv = csv.set_index(csv[0])
    csv = csv.drop(columns=[0])
    dfs.append(csv)

In [7]:
df = pd.concat(dfs)

In [8]:
df = df.sort_index()

In [9]:
df = df.drop_duplicates().reset_index(drop=True) # remove duplicates; export from kontext pretty iffy

In [10]:
df = df.fillna("")

In [11]:
df[3] = df[3].str.replace("-", "")

In [12]:
df["romanian"] = df[2] + " " + df[3] + " " + df[4]

In [13]:
replacements = [
    ("‘ ", "‘"),
    (" ’", "’"),
    (" ,", ","),
    (" .", "."),
    (" :", ":"),
    (" ?", "?"),
    ("( ", "("),
    (" )", ")"),
    (" ;", ";"),
    ("' ", "'"),
    (" '", "'"),
    (" -", "-"),
    ("- ", "-"),
]

In [14]:
for replacement in replacements:
    df["romanian"] = df["romanian"].str.replace(replacement[0], replacement[1])

In [15]:
for replacement in replacements:
    df[7] = df[7].str.replace(replacement[0], replacement[1])

In [16]:
CHUNK_SIZE = 2500

# Parse English

In [13]:
multiprocessing.set_start_method("spawn", force=True)

In [ ]:
process_en = multiprocessing.Process(
    target=functions_stanza.parse_in_chunks, args=(df, "en", 7, "gerunzius", CHUNK_SIZE)
)

In [15]:
process_en.start()

2026-03-03 12:06:19 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-03-03 12:06:20 INFO: Downloaded file to /home/christopher/stanza_resources/resources.json
2026-03-03 12:06:21 INFO: Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined                  |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | combined_charlm           |
| sentiment    | sstplus_charlm            |
| ner          | ontonotes-ww-multi_charlm |

2026-03-03 12:06:21 INFO: Using device: cpu
2026-03-03 12:06:21 INFO: Loading: tokenize
2026-03-03 12:06:21 INFO: Loading: mwt
2026-03-03 12:06:21 INFO: Loading: pos

Parsing chunk 0 to 2499


2026-03-03 12:06:30 INFO: Done loading processors!


In [17]:
parsed_dict_en = functions_stanza.load_pickled_data(folder="gerunzius", language="en")

parsed_slice_en_7500_9999.pkl
parsed_slice_en_10000_11261.pkl
parsed_slice_ro_0_2499.pkl
parsed_slice_en_2500_4999.pkl
parsed_slice_ro_5000_7499.pkl
parsed_slice_ro_10000_11261.pkl
parsed_slice_en_0_2499.pkl
parsed_slice_ro_7500_9999.pkl
parsed_slice_ro_2500_4999.pkl
parsed_slice_en_5000_7499.pkl


In [18]:
adv_part_indices_and_words = []

In [19]:
for i in range(0, len(parsed_dict_en)):
    adv_parts = functions_stanza.get_adv_part_in_sentences(parsed_dict_en[i])
    if adv_parts:
        adv_part_indices_and_words.append((i, adv_parts))

In [20]:
len(adv_part_indices_and_words)

5479

In [21]:
adv_part_indices_and_words

[(5, [('hurrying', 'hurry')]),
 (7, [('wondering', 'wonder')]),
 (8, [('wondering', 'wonder')]),
 (10, [('finding', 'find')]),
 (14, [('muttering', 'mutter')]),
 (15, [('muttering', 'mutter')]),
 (17, [('talking', 'talk')]),
 (22, [('trying', 'try')]),
 (23, [('trying', 'try')]),
 (29, [('licking', 'lick')]),
 (42, [('turning', 'turn')]),
 (43, [('rising', 'rise')]),
 (49, [('pointing', 'point')]),
 (50, [('calling', 'call')]),
 (51, [('turning', 'turn')]),
 (52, [('saying', 'say')]),
 (53, [('looking', 'look')]),
 (54, [('turning', 'turn')]),
 (55, [('turning', 'turn')]),
 (56, [('looking', 'look')]),
 (58, [('getting', 'get')]),
 (59, [('getting', 'get')]),
 (62, [('remarking', 'remark')]),
 (63, [('remarking', 'remark')]),
 (64, [('hoping', 'hope')]),
 (65, [('trotting', 'trot')]),
 (70, [('saying', 'say')]),
 (72, [('taking', 'take')]),
 (73, [('taking', 'take')]),
 (74, [('forgetting', 'forget')]),
 (78, [('scratching', 'scratch'), ('saying', 'say')]),
 (83, [('trying', 'try')]),


# Parse Romanian

In [14]:
multiprocessing.set_start_method("spawn", force=True)

In [15]:
process_ro = multiprocessing.Process(
    target=functions_stanza.parse_in_chunks,
    args=(df, "ro", "romanian", "gerunzius", CHUNK_SIZE),
)

In [ ]:
process_ro.start()

2026-03-09 14:50:46 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-03-09 14:50:46 INFO: Downloaded file to /home/christopher/stanza_resources/resources.json
2026-03-09 14:50:47 INFO: Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined                  |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | combined_charlm           |
| sentiment    | sstplus_charlm            |
| ner          | ontonotes-ww-multi_charlm |

2026-03-09 14:50:47 INFO: Using device: cpu
2026-03-09 14:50:47 INFO: Loading: tokenize
2026-03-09 14:50:48 INFO: Loading: mwt
2026-03-09 14:50:48 INFO: Loading: pos

Parsing chunk 0 to 2499


Parsing chunk 2500 to 4999
Parsing chunk 5000 to 7499
Parsing chunk 7500 to 9999
Parsing chunk 10000 to 11261


In [22]:
parsed_dict_ro = functions_stanza.load_pickled_data(folder="gerunzius", language="ro")

parsed_slice_en_7500_9999.pkl
parsed_slice_en_10000_11261.pkl
parsed_slice_ro_0_2499.pkl
parsed_slice_en_2500_4999.pkl
parsed_slice_ro_5000_7499.pkl
parsed_slice_ro_10000_11261.pkl
parsed_slice_en_0_2499.pkl
parsed_slice_ro_7500_9999.pkl
parsed_slice_ro_2500_4999.pkl
parsed_slice_en_5000_7499.pkl


In [23]:
gerunziu_indices_and_lemmas = []

In [24]:
for i in range(0, len(df)):
    row = df.iloc[i]
    contains_gerunziu, lemma_of_gerunziu = functions_stanza.sentences_contain_gerunziu(
        row[3], parsed_dict_ro[i]
    )
    if contains_gerunziu:
        gerunziu_indices_and_lemmas.append((i, lemma_of_gerunziu))

{
  "id": 14,
  "text": "bombănind",
  "lemma": "bombăni",
  "upos": "VERB",
  "xpos": "Vmg",
  "feats": "VerbForm=Ger",
  "head": 11,
  "deprel": "advcl",
  "start_char": 66,
  "end_char": 75
}
{
  "id": 1,
  "text": "Văzând",
  "lemma": "vedea",
  "upos": "VERB",
  "xpos": "Vmg",
  "feats": "VerbForm=Ger",
  "head": 8,
  "deprel": "advcl",
  "start_char": 102,
  "end_char": 108
}
{
  "id": 3,
  "text": "alunecând",
  "lemma": "aluneca",
  "upos": "VERB",
  "xpos": "Vmg",
  "feats": "VerbForm=Ger",
  "head": 2,
  "deprel": "advcl",
  "start_char": 184,
  "end_char": 193
}
{
  "id": 6,
  "text": "plimbându",
  "lemma": "plimba",
  "upos": "VERB",
  "xpos": "Vmg-------y",
  "feats": "Variant=Short|VerbForm=Ger",
  "head": 5,
  "deprel": "advcl",
  "start_char": 291,
  "end_char": 300
}
{
  "id": 13,
  "text": "întrebând",
  "lemma": "întreba",
  "upos": "VERB",
  "xpos": "Vmg",
  "feats": "VerbForm=Ger",
  "head": 6,
  "deprel": "conj",
  "start_char": 429,
  "end_char": 438
}
{
  "id":

In [25]:
gerunziu_indices_and_lemmas

[(0, 'bombăni'),
 (1, 'vedea'),
 (2, 'aluneca'),
 (3, 'plimba'),
 (4, 'întreba'),
 (5, 'alerga'),
 (6, 'bombăni'),
 (7, 'încerca'),
 (8, 'întrebându'),
 (9, 'trage'),
 (10, 'găsi'),
 (11, 'face'),
 (12, 'întrebându'),
 (13, 'ţina'),
 (14, 'ţinca'),
 (15, 'bombăni'),
 (16, 'apropii'),
 (17, 'spune'),
 (18, 'încerca'),
 (19, 'izbucni'),
 (20, 'băga'),
 (21, 'spune'),
 (22, 'înota'),
 (23, 'sirădui'),
 (24, 'plescăi'),
 (25, 'aminti'),
 (26, 'clipi'),
 (27, 'tremura'),
 (28, 'temandu'),
 (29, 'continua'),
 (30, 'grăbi'),
 (31, 'stârni'),
 (32, 'tremura'),
 (33, 'şiroi'),
 (34, 'simţi'),
 (35, 'vorbi'),
 (36, 'spune'),
 (37, 'forma'),
 (38, 'lua'),
 (39, 'tremura'),
 (40, 'crunta'),
 (41, 'vorbi'),
 (42, 'întoarce'),
 (43, 'ridica'),
 (46, 'gâfâi'),
 (47, 'întreba'),
 (48, 'ţina'),
 (49, 'arăta'),
 (50, 'striga'),
 (51, 'întoarce'),
 (52, 'spune'),
 (53, 'lua'),
 (54, 'întoarce'),
 (55, 'ofta'),
 (56, 'măsura'),
 (57, 'uita'),
 (58, 'ridica'),
 (59, 'îndepărta'),
 (60, 'îndruge'),
 (61, 's

# Building combined df

In [26]:
gerunziu_indices_and_lemmas_df = pd.DataFrame(
    gerunziu_indices_and_lemmas, columns=["index", "lemma"]
).set_index("index")

In [27]:
adv_part_indices_and_words_df = pd.DataFrame(
    adv_part_indices_and_words, columns=["index", "adv_parts_and_lemmas"]
).set_index("index")

In [28]:
gerunziu_indices_and_lemmas_df

,lemma
index,
0,bombăni
1,vedea
2,aluneca
3,plimba
4,întreba
...,...
11257,întune
11258,folosi
11259,ieşi


In [29]:
adv_part_indices_and_words_df

,adv_parts_and_lemmas
index,
5,"[(hurrying, hurry)]"
7,"[(wondering, wonder)]"
8,"[(wondering, wonder)]"
10,"[(finding, find)]"
14,"[(muttering, mutter)]"
...,...
11247,"[(humming, hum)]"
11248,"[(humming, hum)]"
11249,"[(making, make)]"


In [30]:
df = pd.concat(
    [df, gerunziu_indices_and_lemmas_df, adv_part_indices_and_words_df],
    axis=1,
)

In [31]:
df["is_gerunziu"] = False
df.loc[~df["lemma"].isna(), "is_gerunziu"] = True

In [32]:
df.head(2)

,1,2,3,4,5,6,7,8,romanian,lemma,adv_parts_and_lemmas,is_gerunziu
0,carroll-alenka_v_kraji,nu i se păru NEOBIŞNUIT nici chiar atunci când...,bombănind,:,carroll-alenka_v_kraji,,There was nothing so very remarkable in that; ...,,nu i se păru NEOBIŞNUIT nici chiar atunci când...,bombăni,NaN,True
1,carroll-alenka_v_kraji,,Văzând,"una ca asta , Alice se repezi în picioare ;",carroll-alenka_v_kraji,,but when the Rabbit actually took a watch out ...,,"Văzând una ca asta, Alice se repezi în picioare;",vedea,NaN,True


In [33]:
df["has_adv_participle"] = False
df.loc[~df["adv_parts_and_lemmas"].isna(), "has_adv_participle"] = True

In [34]:
len(df.loc[df["is_gerunziu"]])

10881

In [35]:
df.to_csv("extraction_output/annotated_all_ger_to_part.csv")

In [36]:
df.loc[df["is_gerunziu"] & df["has_adv_participle"]].to_csv(
    "extraction_output/gerunziu_with_adv_part.csv"
)

# Assigning gerunzius to adverbial participles

In [37]:
import functions_gensim

In [38]:
df = pd.read_csv("extraction_output/gerunziu_with_adv_part.csv", index_col=0)

In [39]:
for (
    i,
    row,
) in df.iterrows():
    gerunziu = row["3"]
    lemma_ro = row["lemma"]
    adv_parts_and_lemmas = eval(row["adv_parts_and_lemmas"])
    best_match = functions_gensim.get_best_match(
        word=gerunziu,
        lemma=lemma_ro,
        candidate_words_and_lemmas=adv_parts_and_lemmas,
        language_word="ro",
    )
    df.loc[i, ["best_match", "best_score"]] = best_match[0], best_match[1]

In [40]:
df.to_csv("extraction_output/ger_adv_matches.csv")